# Welch's One-Way ANOVA + Games-Howell Post-Hoc Test

This notebook compares the means of three or more independent groups when the population variances may be unequal. Welch's ANOVA is used for the overall test, followed by Games-Howell pairwise comparisons to determine which group means differ.

**Significance level:** α = 0.05


## 1. Why Welch's ANOVA?

Classical one-way ANOVA assumes equal population variances. Welch's ANOVA is a safer alternative when group variances may be unequal and/or sample sizes differ.

**Overall hypotheses:**
- H₀: μ₁ = μ₂ = ... = μₖ
- H₁: At least one population mean is different.

If the Welch ANOVA p-value is below α, reject H₀ and continue with an appropriate post-hoc test.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


## 2. Finance example: daily returns (%)

The groups represent independent trading strategies. The example deliberately uses different levels of volatility.

In [ ]:
strategy_A = np.array([0.08, 0.10, 0.12, 0.09, 0.11, 0.13, 0.07, 0.10, 0.12, 0.09])
strategy_B = np.array([0.10, 0.13, 0.09, 0.15, 0.11, 0.12, 0.08, 0.14, 0.10, 0.13])
strategy_C = np.array([0.20, 0.25, 0.18, 0.31, 0.22, 0.28, 0.16, 0.27, 0.24, 0.30])

groups_data = {
    'Strategy A': strategy_A,
    'Strategy B': strategy_B,
    'Strategy C': strategy_C
}

alpha = 0.05

for name, data in groups_data.items():
    print(f'{name}: n={len(data)}, mean={np.mean(data):.4f}%, SD={np.std(data, ddof=1):.4f}%')


## 3. Welch's one-way ANOVA

Recent SciPy versions support Welch's ANOVA directly through `scipy.stats.f_oneway(..., equal_var=False)`.

In [ ]:
f_stat, p_value = stats.f_oneway(
    strategy_A,
    strategy_B,
    strategy_C,
    equal_var=False
)

print('Welch ANOVA')
print('------------')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value    : {p_value:.6f}')


## 4. Overall ANOVA decision

In [ ]:
if p_value < alpha:
    print('Decision: Reject H0')
    print('There is statistically significant evidence that at least one population mean differs.')
    print('Next step: perform Games-Howell post-hoc pairwise comparisons.')
else:
    print('Decision: Fail to reject H0')
    print('There is insufficient evidence to conclude that the population means differ.')
    print('Post-hoc testing is not normally required after a non-significant overall test.')


## 5. Games-Howell post-hoc test

Games-Howell is commonly paired with Welch's ANOVA because it does not require equal variances and handles unequal sample sizes.

For each pair it evaluates the difference between sample means using a Welch-Satterthwaite degrees-of-freedom calculation and a studentized-range distribution.


In [ ]:
def games_howell(data_dict, alpha=0.05):
    names = list(data_dict.keys())
    k = len(names)
    rows = []

    for i in range(k - 1):
        for j in range(i + 1, k):
            name1, name2 = names[i], names[j]
            x1, x2 = np.asarray(data_dict[name1], dtype=float), np.asarray(data_dict[name2], dtype=float)
            n1, n2 = len(x1), len(x2)
            m1, m2 = np.mean(x1), np.mean(x2)
            v1, v2 = np.var(x1, ddof=1), np.var(x2, ddof=1)

            se2 = v1 / n1 + v2 / n2
            df = (se2 ** 2) / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
            q_stat = abs(m1 - m2) / np.sqrt(se2 / 2)
            p_adj = stats.studentized_range.sf(q_stat, k, df)
            significant = p_adj < alpha

            rows.append({
                'Group 1': name1,
                'Group 2': name2,
                'Mean 1': m1,
                'Mean 2': m2,
                'Mean Difference': m1 - m2,
                'df': df,
                'Q statistic': q_stat,
                'Adjusted p-value': p_adj,
                'Significant': significant
            })

    return pd.DataFrame(rows)


if p_value < alpha:
    gh_results = games_howell(groups_data, alpha=alpha)
    display(gh_results.round(6))
else:
    gh_results = None


## 6. Post-hoc decisions: which groups differ?

In [ ]:
if gh_results is not None:
    for _, row in gh_results.iterrows():
        if row['Significant']:
            print(f"{row['Group 1']} vs {row['Group 2']}: REJECT H0 — means are significantly different (adjusted p={row['Adjusted p-value']:.6f})")
        else:
            print(f"{row['Group 1']} vs {row['Group 2']}: FAIL TO REJECT H0 — insufficient evidence of a difference (adjusted p={row['Adjusted p-value']:.6f})")


## 7. Final interpretation

The overall Welch ANOVA answers: **Is there evidence that at least one population mean differs?**

Games-Howell answers: **Which specific pairs of population means differ?**

A significant pairwise result means there is statistical evidence that the corresponding population means differ. It does not by itself prove that one trading strategy is economically better.

For financial returns, also consider volatility, Sharpe ratio, drawdown, transaction costs, turnover, liquidity, dependence/autocorrelation, and out-of-sample performance.


## 8. Decision workflow

```text
Multiple independent groups
          ↓
Welch's ANOVA
          ↓
     p-value < 0.05?
       /       \
     No         Yes
     ↓           ↓
Fail to       Reject H0
reject H0         ↓
              Games-Howell
                  ↓
        A vs B / A vs C / B vs C
                  ↓
          Which pairs differ?
```


## 9. Important implementation note

The Welch ANOVA call with `equal_var=False` requires a sufficiently recent SciPy version. If your SciPy installation does not recognize this argument, upgrade SciPy before running the notebook.

The Games-Howell implementation above uses SciPy's `studentized_range` distribution, so it does not require a separate post-hoc package.
